# Lab 4 - Is Akisi mi, Ajan mi?

**Kanitladigi tez:** Esneklik bedava degildir. Gorev sabitse is akisi kazanir.

Ayni gorevi iki mimariyle cozup dort olcude karsilastiracagiz:
adim sayisi, token maliyeti, sure, ongorulebilirlik.

## Kurulum

Asagidaki iki hucreyi sirayla calistirin. Bilgisayariniza hicbir sey
kurulmuyor: her sey sizin Colab calisma zamaninizda calisir ve
oturum kapaninca silinir.

In [ ]:
# 1/2 - Repoyu indir
!git clone -q https://github.com/KULLANICI_ADI/guvenli-ai-mimarileri-lab.git 2>/dev/null || echo 'repo zaten var'
%cd -q guvenli-ai-mimarileri-lab
!pip install -q -U transformers accelerate 2>/dev/null
print('kurulum tamam')

In [ ]:
# 2/2 - Modeli sec ve yukle
import os, sys
sys.path.insert(0, '.')

# Uc secenek:
#   'colab' -> kendi calisma zamaninizda kucuk bir model (varsayilan)
#   'mock'  -> model yuklemeden, kayitli cevaplarla (yedek yol)
os.environ['LAB_SAGLAYICI'] = 'colab'

from ortak import llm
print(llm.durum())

# Modeli simdi yukleyelim ki sonraki hucreler beklemesin.
# GPU yoksa bu adim birkac dakika surebilir.
try:
    llm.model_yukle()
except Exception as hata:
    print('Model yuklenemedi:', hata)
    print('MOCK moda geciliyor, lab yapisi aynen calisacak.')
    os.environ['LAB_SAGLAYICI'] = 'mock'

print('\nLab 4 icin hazir.')

---
## Gorev

Bir destek talebini isle: kategorisini bul, alanlarini cikar,
musteriye yanit taslagi yaz.

In [ ]:
TALEP = '''Faturam iki kez kesildi, yarina kadar cozulmezse
iptal etmek istiyorum. Agustos donemi, yaklasik 12.400 TL fazla.
Musteri no: M-1042'''

print(TALEP)

---
## 1. Is akisi: adimlari SIZ yaziyorsunuz

Uc dar adim. Her adim tek is yapar, her adim ayri test edilebilir.

In [ ]:
from ortak import llm
import time

def is_akisi(talep):
    olcum = {'adim': 0, 'token': 0, 'sure': 0.0}
    baslangic = time.time()

    # Adim 1: siniflandirma
    c1 = llm.sor(f'Bu talebin kategorisini JSON olarak dondur '
                 f'("kategori" alani):\n{talep}',
                 sicaklik=0.0, senaryo='lab4_sinif')
    kategori = llm.json_ayikla(c1.metin) or {}
    olcum['adim'] += 1; olcum['token'] += c1.toplam_token

    # Adim 2: alan cikarimi
    c2 = llm.sor(f'Bu talepten donem, tutar ve musteri_id alanlarini '
                 f'JSON olarak cikar:\n{talep}',
                 sicaklik=0.0, senaryo='lab4_alan')
    alanlar = llm.json_ayikla(c2.metin) or {}
    olcum['adim'] += 1; olcum['token'] += c2.toplam_token

    # Adim 3: yanit taslagi
    c3 = llm.sor(f'Su bilgilerle kisa bir musteri yaniti yaz: '
                 f'{kategori} {alanlar}',
                 sicaklik=0.3, senaryo='lab4_yanit')
    olcum['adim'] += 1; olcum['token'] += c3.toplam_token

    olcum['sure'] = time.time() - baslangic
    return {'kategori': kategori, 'alanlar': alanlar,
            'yanit': c3.metin, 'olcum': olcum}

sonuc_akis = is_akisi(TALEP)
print('Kategori:', sonuc_akis['kategori'])
print('Alanlar :', sonuc_akis['alanlar'])
print('Yanit   :', sonuc_akis['yanit'][:120])
print()
print('Olcum:', sonuc_akis['olcum'])

---
## 2. Ajan: adimlari MODEL seciyor

Ayni gorev, ama bu kez model hangi araci ne zaman cagiracagina
ve ne zaman duracagina kendi karar veriyor.

In [ ]:
SAHTE_VERI = {
    'talep_getir': {'talep_id': 'T-77', 'metin': TALEP},
    'musteri_getir': {'musteri_id': 'M-1042', 'durum': 'aktif'},
    'fatura_getir': {'donem': '2026-08', 'tutar': 12400},
}

def ajan(talep, azami_tur=8):
    olcum = {'adim': 0, 'token': 0, 'sure': 0.0}
    baslangic = time.time()
    gecmis = [f'Gorev: {talep}']

    for tur in range(azami_tur):
        c = llm.sor('\n'.join(gecmis) + '\n\nSirada ne yapiyorsun?',
                    sicaklik=0.3, senaryo='lab4_ajan')
        olcum['token'] += c.toplam_token
        karar = llm.json_ayikla(c.metin)
        if not karar:
            break

        if 'cevap' in karar:
            olcum['sure'] = time.time() - baslangic
            return {'yanit': karar['cevap'], 'olcum': olcum}

        arac = karar.get('arac')
        olcum['adim'] += 1
        print(f"  tur {tur+1}: {karar.get('dusunce','')} -> {arac}")
        sonuc = SAHTE_VERI.get(arac, {'hata': 'bilinmeyen arac'})
        gecmis.append(f'{arac} sonucu: {sonuc}')

    olcum['sure'] = time.time() - baslangic
    return {'yanit': None, 'olcum': olcum}

sonuc_ajan = ajan(TALEP)
print()
print('Yanit:', (sonuc_ajan['yanit'] or '(tamamlanmadi)')[:120])
print('Olcum:', sonuc_ajan['olcum'])

> Ajanin turlerine dikkat edin. Ayni araci birden fazla kez
> cagirdigini goruyor musunuz? Bu, kontrolsuz dongunun en hafif halidir.

---
## 3. Karsilastirma

In [ ]:
a, b = sonuc_akis['olcum'], sonuc_ajan['olcum']

print(f"{'Olcu':<20} {'Is akisi':>12} {'Ajan':>12}")
print('-' * 46)
print(f"{'Adim sayisi':<20} {a['adim']:>12} {b['adim']:>12}")
print(f"{'Token':<20} {a['token']:>12} {b['token']:>12}")
print(f"{'Sure (sn)':<20} {a['sure']:>12.2f} {b['sure']:>12.2f}")

---
## 4. Ongorulebilirlik: asil fark burada

Tek calistirma yaniltici olabilir. Ikisini de bes kez calistirip
**sonuclarin dagilimina** bakalim.

In [ ]:
print('Is akisi, 5 calistirma:')
akis_adimlar = []
for i in range(5):
    s = is_akisi(TALEP)
    akis_adimlar.append(s['olcum']['adim'])
print(' adim sayilari:', akis_adimlar)

print()
print('Ajan, 5 calistirma:')
ajan_adimlar = []
for i in range(5):
    s = ajan(TALEP)
    ajan_adimlar.append(s['olcum']['adim'])
print(' adim sayilari:', ajan_adimlar)

print()
print(f'Is akisi araligi: {min(akis_adimlar)}-{max(akis_adimlar)}')
print(f'Ajan araligi    : {min(ajan_adimlar)}-{max(ajan_adimlar)}')
print()
print('Bir sistemi butcelemek icin ust siniri bilmeniz gerekir.')

---
## 5. Ajani uretime hazirlayan sey: sinirlar

Ajan kotu degil, **sinirsiz ajan** kotudur. Uc sinir ekleyelim.

In [ ]:
def sinirli_ajan(talep, azami_tur=4, azami_token=2000,
                 azami_ayni_arac=1):
    """Adim siniri + token butcesi + ayni araci tekrar cagirma siniri."""
    olcum = {'adim': 0, 'token': 0, 'durdurma': None}
    gecmis = [f'Gorev: {talep}']
    arac_sayaci = {}

    for tur in range(azami_tur):
        if olcum['token'] > azami_token:
            olcum['durdurma'] = 'TOKEN BUTCESI ASILDI'
            break

        c = llm.sor('\n'.join(gecmis) + '\n\nSirada ne yapiyorsun?',
                    sicaklik=0.3, senaryo='lab4_ajan')
        olcum['token'] += c.toplam_token
        karar = llm.json_ayikla(c.metin)
        if not karar:
            olcum['durdurma'] = 'AYRISTIRILAMADI'
            break
        if 'cevap' in karar:
            olcum['durdurma'] = 'NORMAL BITIS'
            break

        arac = karar.get('arac')
        arac_sayaci[arac] = arac_sayaci.get(arac, 0) + 1
        if arac_sayaci[arac] > azami_ayni_arac:
            olcum['durdurma'] = f'TEKRAR DONGUSU: {arac}'
            break

        olcum['adim'] += 1
        gecmis.append(f'{arac} sonucu: {SAHTE_VERI.get(arac, {})}')
    else:
        olcum['durdurma'] = 'ADIM SINIRI'

    return olcum

for i in range(3):
    print(sinirli_ajan(TALEP))

---
## Egzersiz

1. `azami_tur` degerini 2 yapin. Ajan gorevi tamamlayabiliyor mu?
2. Is akisina dorduncu bir adim ekleyin. Token maliyeti nasil degisti?
3. Kendi isinizden bir gorev secin ve sorun: adimlari onceden
   yazabiliyor musunuz? Yaziyorsaniz ajana ihtiyaciniz yok.

---
## Alinacak ders

> Ajan bir varsayilan degil, son caredir. Once is akisini deneyin.
> Ajan kullanacaksaniz adim siniri, token butcesi ve tekrar dongusu
> korumasi olmadan uretime cikarmayin.